# Optimizations using Krotov's method for quantum control

```2026/05/06```

1. Review theory for Krotov and write it here;
2. Follow pseudocode;
3. Compare to ```krotov``` package.


List of comparisons:

1. Effect of optimization parameters (state to state)
    - Total time (100, 500);
    - Time step  (0.05, 0.001);
    - $\lambda_a$. (50, 25)
2. Effect of initial field (state to state)
    - Constant;
    - Sinusoidal;
    - Large pulse;
    - Quasi optiimal (sinusoidal with small amplitude and frequency adjusted to the *gap*).
3. Effect of system parameter (state to state):
    - Different values of $\alpha$.
4. Showcase: multi objective optimization;
5. Showcase: ionization optimization.

* Finally: apply most interesting fields to classical MsC.

In [1]:
from emerald.quantum.msc_unperturbed import MsC_hamiltonian, MsC_eigstates
from emerald.quantum.coupling_utils import interaction_matrix
from emerald.quantum.krotov import (
    KrotovOptimizer,
    KrotovMultiOptimizer,
    IonizationOptimizer,
    single_step_evolution,
    single_step_inverse_evolution,
    S_l,
    Objective
)
from emerald.utils import FieldParams, external_field_array
import numpy as np
import json
import qutip
import krotov
from datetime import datetime
import time


def calculate_specter(alpha, r_min, r_max, N_points, base_size):

    # Position grid
    r_grid = np.linspace(r_min, r_max, N_points)

    # Hamiltonian and eigenstates
    H, r_grid = MsC_hamiltonian(alpha, r_grid)

    energies, eigstates = MsC_eigstates(H, r_grid, base_size)

    # Interaction matrix
    Xi_vecs, Xi_vals, Xi_conjT = interaction_matrix(r_grid, eigstates)

    return energies, eigstates, Xi_vecs, Xi_vals, Xi_conjT


def H0_propagator(energies, delta_t):
    exp_H0_fwd = np.exp(-1j * energies * delta_t / 2)
    exp_H0_bwd = np.exp(1j * energies * delta_t / 2)
    return exp_H0_fwd, exp_H0_bwd


def pure_basis_state(base_size, state_index):
    state = np.zeros(base_size)
    state[state_index] = 1.0
    return state


def split_op_propagator(H, state, dt, c_ops=None, backwards=False, initialize=False):
    """
    Wraps split-operator stepper for the krotov package.
    H here is the assembled Hamiltonian at this time step as a Qobj.
    The package calls it as H = H0 + eps*H1, but we bypass that and use
    our pre-factored form for efficiency.
    """
    eps = float(np.real(H[1][1]))  # scalar control value baked in by krotov
    psi = state.full().ravel()  # Qobj -> numpy 1D array

    if not backwards:
        result = single_step_evolution(
            psi, eps, exp_H0_fwd, Xi_vals, Xi_vecs, Xi_conjT, dt
        )
    else:
        result = single_step_inverse_evolution(
            psi, eps, exp_H0_bwd, Xi_vals, Xi_vecs, Xi_conjT, dt
        )

    return qutip.Qobj(result)


def convert_times(start_local_time, end_local_time):
    start_dt = datetime.fromtimestamp(time.mktime(start_local_time))
    end_dt = datetime.fromtimestamp(time.mktime(end_local_time))

    timestamp = (start_dt.isoformat(),)
    elapsed = (end_dt - start_dt).total_seconds()

    return timestamp, elapsed


def dict_from_krotov_result(opt_result_krotov):

    start_local_time = opt_result_krotov.start_local_time
    end_local_time = opt_result_krotov.end_local_time

    timestamp, elapsed = convert_times(start_local_time, end_local_time)

    _dict = {
        "metadata": {
            "timestamp": timestamp[0],
            "objectives": list(
                [opt.summarize() for opt in opt_result_krotov.objectives]
            ),
            "n_iterations": len(opt_result_krotov.iters) - 1,
            "elapsed_time_seconds": elapsed,
        },
        "results": {
            "best J_T": np.min(opt_result_krotov.info_vals),
            "best_iteration": float(np.argmin(opt_result_krotov.info_vals)),
            "final_overlap": (np.abs(opt_result_krotov.tau_vals[-1]) ** 2)[0],
            "J_T_history": list(opt_result_krotov.info_vals),
            "optimized_field": list(opt_result_krotov.optimized_controls[0]),
        },
        "per objective": None,
    }
    return _dict

Now it is convenient to define the time array and pre calculate a few things:

In [2]:
## Cell to calculate parameters that dont change often

# Parameters for the specter calculation
alpha = 1.0
r_min, r_max = -3.5, 1000
N_points = 5000
base_size = 500

energies, eigstates, Xi_vecs, Xi_vals, Xi_conjT = calculate_specter(
    alpha, r_min, r_max, N_points, base_size
)

In [3]:
## Prototypical cell to run and save optimizations

# Third: sine field, 0 -> 1 transition,
# frequency set to oscilate 10 times

opt_number = 1
myopt = True

# State definitions
initial_state = pure_basis_state(base_size, 0)  # Ground state
target_state1 = pure_basis_state(base_size, 1)  # First excited state
target_state2 = pure_basis_state(base_size, 2)  # First excited state

# Time grid for optimization
T = 100.0
delta_t = 0.05
time_grid = np.arange(0, T + delta_t, delta_t)

# Initial guess for the control field

field_params = FieldParams(
    amplitude=0.5,
    frequency= (energies[2]+energies[1])/2 - energies[0] , 
    envelope="linear",
    rampup_time=10,
    rampdown_time=10,
    form="sin",
    operation_time=T,
)

initial_guess_field = external_field_array(time_grid, field_params)


# Optimization parameters
lambda_a = 10
t_on = 10
t_off = 10
max_iter = 500
JT_thresh = 5e-3

if myopt:
    # Run the Krotov optimization

    # --- Multi-objective (ground → excited1 AND ground → excited2) ---
    objectives = [
        Objective(psi_0=initial_state, psi_target=target_state1, weight=0.5),
        Objective(psi_0=initial_state, psi_target=target_state2, weight=0.5),
    ]
    optimizer = KrotovMultiOptimizer(
        objectives=objectives,
        energies=energies, 
        Xi_vals=Xi_vals, 
        Xi_vecs=Xi_vecs, 
        Xi_conjT=Xi_conjT,
        time_grid=time_grid, 
        initial_field=initial_guess_field,
        lambda_a=lambda_a, 
        t_on=t_on,
        t_off=t_off,
        max_iterations=max_iter,
        JT_threshold=JT_thresh,
        store_histories=False,
    )

    opt_result_mine = optimizer.run()
    opt_result_mine_dict = opt_result_mine.to_dict()

    opt_result_mine_dict.update(
        {
            "General parameters": {
                "initial_field": list(initial_guess_field),
                "alpha": alpha,
                "r_min": r_min,
                "r_max": r_max,
                "N_points": N_points,
                "base_size": base_size,
                "total_time": T,
                "delta_t": delta_t,
                "lambda_a": lambda_a,
                "t_on": t_on,
                "t_off": t_off,
                "max_iter": max_iter,
                "JT_thresh": JT_thresh,
                "Transition": "0 -> [1, 2]",
            }
        }
    )

    filename = (
        f"MyOpt{opt_number:02d}--{opt_result_mine_dict['metadata']['timestamp']}.json"
    )
    with open(filename, "w") as filehandle:
        json.dump(opt_result_mine_dict, filehandle, indent=4)

iter.           J_T     delta J_T    secs
-------------------------------------------------------
    0  9.607473e-01           n/a     0.0
    1  9.662505e-01    +5.503e-03     7.4
    2  9.198167e-01    -4.643e-02     7.4
    3  8.902975e-01    -2.952e-02     6.9
    4  8.402744e-01    -5.002e-02     9.8
    5  8.126726e-01    -2.760e-02     7.8
    6  7.830490e-01    -2.962e-02     7.8
    7  7.633915e-01    -1.966e-02     7.6
    8  7.457683e-01    -1.762e-02     7.8
    9  7.320361e-01    -1.373e-02     7.9
   10  7.204407e-01    -1.160e-02     7.8
   11  7.101710e-01    -1.027e-02     7.6
   12  7.015741e-01    -8.597e-03     7.4
   13  6.933717e-01    -8.202e-03     7.2
   14  6.864599e-01    -6.912e-03     7.1
   15  6.796304e-01    -6.830e-03     7.3
   16  6.738088e-01    -5.822e-03     7.8
   17  6.679727e-01    -5.836e-03     7.2
   18  6.629309e-01    -5.042e-03     6.8
   19  6.578535e-01    -5.077e-03     7.1
   20  6.534060e-01    -4.448e-03     7.3
   21  6.489274e-01 

In [4]:
## Prototypical cell to run and save optimizations

# Third: sine field, 0 -> 1 transition,
# frequency set to oscilate 10 times

opt_number = 2
myopt = True

# State definitions
initial_state = pure_basis_state(base_size, 0)  # Ground state

# Time grid for optimization
T = 100.0
delta_t = 0.05
time_grid = np.arange(0, T + delta_t, delta_t)

# Initial guess for the control field

initial_guess_field = S_l(time_grid, T, T/4, T/4)

# Optimization parameters
lambda_a = 50
t_on = 10
t_off = 10
max_iter = 300
JT_thresh = 5e-3

if myopt:
    # Run the Krotov optimization

    bound_mask = energies < 0
    optimizer = IonizationOptimizer(
        psi_0=initial_state, 
        bound_mask=bound_mask,
        energies=energies, 
        Xi_vals=Xi_vals, 
        Xi_vecs=Xi_vecs, 
        Xi_conjT=Xi_conjT,
        time_grid=time_grid, 
        initial_field=initial_guess_field,
        lambda_a=lambda_a, 
        t_on=t_on,
        t_off=t_off,
        max_iterations=max_iter,
        JT_threshold=JT_thresh,
        store_histories=False,
    )

    opt_result_mine = optimizer.run()
    opt_result_mine_dict = opt_result_mine.to_dict()

    opt_result_mine_dict.update(
        {
            "General parameters": {
                "initial_field": list(initial_guess_field),
                "alpha": alpha,
                "r_min": r_min,
                "r_max": r_max,
                "N_points": N_points,
                "base_size": base_size,
                "total_time": T,
                "delta_t": delta_t,
                "lambda_a": lambda_a,
                "t_on": t_on,
                "t_off": t_off,
                "max_iter": max_iter,
                "JT_thresh": JT_thresh,
                "Transition": "0 -> [1, 2]",
            }
        }
    )

    filename = (
        f"MyOpt{opt_number:02d}--{opt_result_mine_dict['metadata']['timestamp']}.json"
    )
    with open(filename, "w") as filehandle:
        json.dump(opt_result_mine_dict, filehandle, indent=4)

iter.     J_T (bound)   unbound pop     delta J_T    secs
-------------------------------------------------------------
    0  9.992322e-01    0.000768           n/a     0.0
    1  9.991766e-01    0.000823    -5.558e-05     3.4
    2  9.991256e-01    0.000874    -5.106e-05     3.5
    3  9.990710e-01    0.000929    -5.453e-05     3.8
    4  9.990128e-01    0.000987    -5.824e-05     3.6
    5  9.989506e-01    0.001049    -6.219e-05     3.5
    6  9.988842e-01    0.001116    -6.641e-05     3.8
    7  9.988133e-01    0.001187    -7.092e-05     4.1
    8  9.987375e-01    0.001262    -7.574e-05     3.8
    9  9.986567e-01    0.001343    -8.088e-05     3.9
   10  9.985703e-01    0.001430    -8.637e-05     3.9
   11  9.984780e-01    0.001522    -9.224e-05     3.6
   12  9.983795e-01    0.001620    -9.850e-05     3.5
   13  9.982743e-01    0.001726    -1.052e-04     3.8
   14  9.981620e-01    0.001838    -1.123e-04     3.9
   15  9.980420e-01    0.001958    -1.200e-04     3.7
   16  9.979139e

In [5]:
## Prototypical cell to run and save optimizations

# Third: sine field, 0 -> 1 transition,
# frequency set to oscilate 10 times

opt_number = 3
myopt = True

# State definitions
initial_state = pure_basis_state(base_size, 0)  # Ground state

# Time grid for optimization
T = 100.0
delta_t = 0.05
time_grid = np.arange(0, T + delta_t, delta_t)

# Initial guess for the control field

bound_states = np.sum(energies<0)

field_params = FieldParams(
    amplitude=0.1,
    frequency=(energies[bound_states] - energies[0]), 
    envelope="linear",
    rampup_time=10,
    rampdown_time=10,
    form="sin",
    operation_time=T,
)

initial_guess_field = external_field_array(time_grid, field_params)

# Optimization parameters
lambda_a = 50
t_on = 10
t_off = 10
max_iter = 300
JT_thresh = 5e-3

if myopt:
    # Run the Krotov optimization

    bound_mask = energies < 0
    optimizer = IonizationOptimizer(
        psi_0=initial_state, 
        bound_mask=bound_mask,
        energies=energies, 
        Xi_vals=Xi_vals, 
        Xi_vecs=Xi_vecs, 
        Xi_conjT=Xi_conjT,
        time_grid=time_grid, 
        initial_field=initial_guess_field,
        lambda_a=lambda_a, 
        t_on=t_on,
        t_off=t_off,
        max_iterations=max_iter,
        JT_threshold=JT_thresh,
        store_histories=False,
    )

    opt_result_mine = optimizer.run()
    opt_result_mine_dict = opt_result_mine.to_dict()

    opt_result_mine_dict.update(
        {
            "General parameters": {
                "initial_field": list(initial_guess_field),
                "alpha": alpha,
                "r_min": r_min,
                "r_max": r_max,
                "N_points": N_points,
                "base_size": base_size,
                "total_time": T,
                "delta_t": delta_t,
                "lambda_a": lambda_a,
                "t_on": t_on,
                "t_off": t_off,
                "max_iter": max_iter,
                "JT_thresh": JT_thresh,
                "Transition": "0 -> [1, 2]",
            }
        }
    )

    filename = (
        f"MyOpt{opt_number:02d}--{opt_result_mine_dict['metadata']['timestamp']}.json"
    )
    with open(filename, "w") as filehandle:
        json.dump(opt_result_mine_dict, filehandle, indent=4)

iter.     J_T (bound)   unbound pop     delta J_T    secs
-------------------------------------------------------------
    0  7.014370e-01    0.298563           n/a     0.0
    1  2.848338e-01    0.715166    -4.166e-01     5.1
    2  2.609933e-01    0.739007    -2.384e-02     4.9
    3  2.406742e-01    0.759326    -2.032e-02     5.5
    4  2.230315e-01    0.776969    -1.764e-02     4.9
    5  2.075653e-01    0.792435    -1.547e-02     3.7
    6  1.939643e-01    0.806036    -1.360e-02     3.6
    7  1.819459e-01    0.818054    -1.202e-02     3.6
    8  1.712435e-01    0.828757    -1.070e-02     3.2
    9  1.616161e-01    0.838384    -9.627e-03     4.0
   10  1.528616e-01    0.847138    -8.754e-03     8.4
   11  1.448204e-01    0.855180    -8.041e-03     3.7
   12  1.373700e-01    0.862630    -7.450e-03     3.7
   13  1.304182e-01    0.869582    -6.952e-03     4.2
   14  1.238950e-01    0.876105    -6.523e-03     4.6
   15  1.177476e-01    0.882252    -6.147e-03     4.2
   16  1.119354e

In [6]:
import os

os.system("shutdown +1")

Shutdown scheduled for Sun 2026-05-10 01:35:04 -03, use 'shutdown -c' to cancel.


0